# W12 Practice — An MCP Server of Our Own

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week12/W12_practice_mcp_server.ipynb)

**Goal.** Build an MCP server over a web API, use it from an agent, and define its tools by five principles of tool design. One question runs on two servers over the same API: first a server that wraps each request of the API as a tool, then a server whose tools follow the principles.

| Part | What you learn | Main names |
|---|---|---|
| 1. The recipe API | two requests to a web API | `httpx2` |
| 2. A server that wraps the API | one tool for each request; the tools in an agent | `MCPServer`, `@mcp.tool`, `MCPAdapter` |
| 3. Five principles of tool design | what a model needs from a tool | |
| 4. A server built by the principles | five tools for the same question | `ToolError`, `Literal` |
| 5. Exercises | three more tools | |

**Cell labels.** The label is the first line of a code cell, or the second line when the cell saves a file with `%%writefile`:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.
- `# ✍️ TRY · exercise`: write your code in the cell, then run (Part 5).

## An MCP Server over a Web API

An MCP server gives a web API the form of tools: each tool is a function that sends a request to the API and returns the reply. Three programs take part.

| Program | What it does | Where |
|---|---|---|
| Web API | keeps the recipes and answers HTTP requests | TheMealDB, on the internet |
| MCP server | offers tools; each tool calls the web API | a file in this runtime |
| MCP host | gives the tools of the server to an agent | this notebook |

A server in Python is written in five steps.

| Step | Code |
|---|---|
| 1. Make the server object | `mcp = MCPServer("recipes")` |
| 2. Write one function for each tool | `@mcp.tool()` above the function |
| 3. Call the web API in the function and return the result | `httpx2.get(...)` |
| 4. Run the server | `mcp.run()` |
| 5. Connect a client and get the tools | `MCPAdapter(PythonStdioTransport("server.py"))`, `list_tools()` |

The server makes the definition of a tool from the function: the function name becomes the tool name, the type hints the arguments, and the docstring the description. The client starts the server file as a program and talks to it through its standard input and output (the **stdio** transport of MCP).

Source of the steps: MCP documentation, *Build an MCP server*.

## 0. Setup

*Do:* run the two cells; paste your API key into the second.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain[mcp]==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os
from langchain.chat_models import init_chat_model

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

MODEL = "openai:gpt-5.4"
model = init_chat_model(MODEL)                     # one model object for the whole notebook

## 1. The Recipe API

TheMealDB is a free database of recipes with a web API that needs no sign-up. Two of its requests are used first.

| Request | What it returns |
|---|---|
| `GET /filter.php?i=<ingredient>` | the meals that use one ingredient |
| `GET /lookup.php?i=<meal ID>` | the recipe of one meal |

*Do:* run the cell. It sends both requests and prints the replies.

In [ ]:
# ▶ RUN · example: read each line, then run
import httpx2                                      # an HTTP client for Python; it came with the packages of the setup

API = "https://www.themealdb.com/api/json/v1/1"    # the address of the API; "1" is the public test key

reply = httpx2.get(API + "/filter.php", params={"i": "chicken breast"})   # the meals that use chicken breast
meals = reply.json()["meals"]                      # .json() turns the JSON of the reply into Python lists and dicts
print(len(meals), "meals; the first one:", meals[0])

reply = httpx2.get(API + "/lookup.php", params={"i": meals[0]["idMeal"]})   # the recipe of the first meal, by its ID
print(reply.text)

**Reading the output.** A search returns the name, a picture link, the ID, and the country of each meal, for one ingredient only. A recipe needs a second request; its reply has 54 fields, among them twenty pairs `strIngredient1` to `strIngredient20` and `strMeasure1` to `strMeasure20`, and the pairs that a meal does not use are empty.

## 2. A Server that Wraps the API

The first server has one tool for each request and returns the reply of the API as it comes.

*Do:* run the cell. It saves the server as `meal_mcp_v1.py`.

In [ ]:
%%writefile meal_mcp_v1.py
# ▶ RUN · example: read each line, then run
import httpx2
from mcp.server import MCPServer

API = "https://www.themealdb.com/api/json/v1/1"

## Step 1 - the server object
mcp = MCPServer("recipes", log_level="WARNING")    # the name that the server gives to its clients; log only warnings


## Steps 2 and 3 - one function for each tool; the function calls the web API
@mcp.tool()
def search_by_ingredient(ingredient: str) -> str:
    """Find the meals that use one ingredient.

    Args:
        ingredient: One ingredient, for example "chicken breast".
    """
    return httpx2.get(API + "/filter.php", params={"i": ingredient}).text   # the reply of the API is the tool result


@mcp.tool()
def get_recipe(meal_id: str) -> str:
    """Get the recipe of one meal.

    Args:
        meal_id: The ID of the meal, for example "52772".
    """
    return httpx2.get(API + "/lookup.php", params={"i": meal_id}).text


## Step 4 - run the server
if __name__ == "__main__":                         # true when this file is started as a program
    mcp.run()                                      # stdio: read requests from standard input, write replies to standard output

### 2.2 The question on this server

*Do:* read `QUESTION`, then run the cell. It connects to the server (step 5), gives its tools to an agent, and prints the path of the run: each tool call, the length of each tool result, and the answer.

In [ ]:
# ▶ RUN · example: read each line, then run
from pathlib import Path
from fastmcp.client.transports import PythonStdioTransport
from langchain.mcp import MCPAdapter               # this import prints one warning: this part of LangChain is in beta
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def show(result):
    """Print the path of one run: the tool calls, the length of each tool result, and the answer."""
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:        # one AI message can ask for several tools
                print("TOOL CALL  :", call["name"], call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.name, "-", len(message.text), "characters")
    print()
    print(result["messages"][-1].text)             # the last message is the answer

## Step 5 - connect a client and get the tools
server_v1 = PythonStdioTransport("meal_mcp_v1.py", log_file=Path("server.log"))   # start the file as a program, talk over stdio;
                                                   # its messages go to server.log (Colab has no place to show them)
async with MCPAdapter(server_v1) as adapter:
    tools_v1 = await adapter.list_tools()

agent_v1 = create_agent(model, tools_v1)

QUESTION = "닭가슴살과 마늘만 있어. 이 두 재료가 모두 들어가는 요리 두 개를 골라 재료와 조리 순서, 레시피 전체의 칼로리, 그리고 더 사야 할 재료를 알려줘."
result = await agent_v1.ainvoke({"messages": [HumanMessage(QUESTION)]})   # the tools are asynchronous, so the agent runs with ainvoke
show(result)

**Reading the output.** In our run the model made four calls. It searched for chicken breast and for garlic, one call each, and the garlic search returned about 42,000 characters. It compared the two lists itself and asked for two recipes. The server has no tool for calories or for a shopping list, so the model wrote both from what it knows: the calories are ranges such as 3,300 to 3,500 kcal, and the shopping list is the ingredient list of each recipe without chicken breast and garlic.

## 3. Five Principles of Tool Design

A request of an API is designed for a program, which reads any amount of JSON at once. A tool is read by a model, which reads every result as text in a context of limited size and chooses tools by their descriptions. Anthropic's guide for writers of tools states five principles for this reader. The MCP specification fixes the format of a tool (name, description, input schema, result, errors) but not its design; three of its rules support the principles and are in the table.

| Principle | What it asks | The server of Part 2 | The server of Part 4 |
|---|---|---|---|
| 1. One tool for one task of the user | a tool does what a person wants done, not one request of the API | two searches and a comparison of the lists by the model; no tool for the calories or the shopping list, so the model estimated them itself | `recipe_find_by_ingredients` takes several ingredients and compares the lists; `recipe_count_calories` and `recipe_shopping_list` do the rest of the question |
| 2. Names that tell tools apart | an agent gets the tools of many servers; a common prefix groups the tools of one service. MCP: a name has 1 to 128 letters, digits, `_`, `-`, or `.` | `search_by_ingredient`, `get_recipe` | every tool begins with `recipe_` |
| 3. Only what the next step needs | the result holds the information that the model uses next, in a limited amount | a search returns a picture link for each meal; a recipe has 54 fields, the empty ones too | name, country, and ID; a recipe without empty fields; at most `limit` meals, with "... and N more" |
| 4. Errors that say how to correct the call | an error says what is wrong and what to send instead. MCP: a tool execution error goes back to the model with `isError`, so that it can correct its call | an unknown ingredient returns `{"meals":null}` | `ToolError("No meal uses 'X'. Write the ingredient in English, ...")` |
| 5. Descriptions and inputs without ambiguity | the description says when to use the tool and what each argument means, as to a new colleague; a strict schema enforces the input | one sentence; any text as argument | when to use each tool; the food of `recipe_count_calories` must be one of the 78 foods of a calorie table |

The MCP specification also lets a server mark a tool, for example as read-only (`annotations`). The tools of Part 4 carry this mark; a host can use it to decide which calls a person must approve.

Sources: Anthropic, *Writing effective tools for agents* (2025); MCP specification, *Tools* (2026-07-28).

## 4. A Server Built by the Principles

The second server answers the same question with five tools. The comment above each tool names the principles that it follows.

The calories come from a table of 78 common foods with their energy per 100 g, taken from the USDA food composition database. The model reads the recipe, chooses the closest food of the table for each ingredient, and converts each measure into grams in the arguments of the call; the server looks up the table and adds. The calories are therefore an estimate.

*Do:* run the two cells. The first saves `meal_mcp_v2.py`; the second adds two tools to the end of the file (`%%writefile -a`).

In [ ]:
%%writefile meal_mcp_v2.py
# ▶ RUN · example: read each line, then run
import csv
from typing import Literal

import httpx2
from pydantic import BaseModel
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from mcp.types import ToolAnnotations

API = "https://www.themealdb.com/api/json/v1/1"
mcp = MCPServer("recipes", log_level="WARNING")
READ_ONLY = ToolAnnotations(read_only_hint=True)   # the mark "this tool only reads"

KCAL = {}                                          # standard food -> kcal per 100 g, from the USDA table
for row in csv.DictReader(open("food_calories.csv", encoding="utf-8")):
    KCAL[row["food"]] = float(row["kcal_per_100g"])
Food = Literal[tuple(KCAL)]                        # a type that allows only the 78 names of the table


def get_meals(path, params):
    """Send one request to the API and return its list of meals ([] when there is none). Not a tool."""
    meals = httpx2.get(API + path, params=params).json()["meals"]
    if meals is None:
        return []
    return meals


## Principles 1, 2, 3, 4: several ingredients in one call, the prefix recipe_, a limit, errors that say what to do
@mcp.tool(annotations=READ_ONLY)
def recipe_find_by_ingredients(ingredients: list[str], limit: int = 10) -> str:
    """Find meals that use ALL of the given ingredients.
    Use this when the user names ingredients that they have or want to use.
    Returns one line per meal: the name, the country, and the meal ID.

    Args:
        ingredients: One or more ingredients in English, for example ["chicken breast", "garlic"].
        limit: The largest number of meals to return.
    """
    found = None                                   # meal ID -> "name (country)", for the meals in every list
    for ingredient in ingredients:                 # the API takes one ingredient per request
        meals = get_meals("/filter.php", {"i": ingredient})
        if not meals:
            raise ToolError("No meal uses '" + ingredient + "'. Write the ingredient in English, for example 'chicken breast'.")
        names = {}
        for meal in meals:
            names[meal["idMeal"]] = meal["strMeal"] + " (" + str(meal["strCountry"]) + ")"
        if found is None:
            found = names
        else:
            both = {}
            for meal_id in found:
                if meal_id in names:               # keep only the meals that are in both lists
                    both[meal_id] = found[meal_id]
            found = both
    if not found:
        raise ToolError("No meal uses all of: " + ", ".join(ingredients) + ". Try fewer ingredients.")
    lines = [str(len(found)) + " meals use " + ", ".join(ingredients) + ":"]
    for meal_id in list(found)[:limit]:
        lines.append("- " + found[meal_id] + ", ID " + meal_id)
    if len(found) > limit:
        lines.append("... and " + str(len(found) - limit) + " more. Add an ingredient to narrow the list.")
    return "\n".join(lines)


## Principles 3, 4, 5: no empty fields, an error that says what to do, when to use the tool
@mcp.tool(annotations=READ_ONLY)
def recipe_get_recipe(meal_id: str) -> str:
    """Get the recipe of one meal: its ingredients with their measures, and the instructions.
    Use a meal ID from recipe_find_by_ingredients.

    Args:
        meal_id: The ID of the meal, for example "52818".
    """
    meals = get_meals("/lookup.php", {"i": meal_id})
    if not meals:
        raise ToolError("No meal has the ID " + meal_id + ". Use an ID from recipe_find_by_ingredients.")
    meal = meals[0]
    lines = [meal["strMeal"] + " (" + str(meal["strCountry"]) + ")", "Ingredients:"]
    for i in range(1, 21):                         # the API keeps twenty numbered pairs
        name = meal["strIngredient" + str(i)]
        if name:                                   # skip the empty pairs
            lines.append("- " + str(meal["strMeasure" + str(i)]) + " " + name)
    lines.append("Instructions:")
    lines.append(meal["strInstructions"])
    return "\n".join(lines)

In [ ]:
%%writefile -a meal_mcp_v2.py
# ▶ RUN · example: read each line, then run


class FoodWeight(BaseModel):                       # one ingredient, as the model must send it
    ingredient: str                                # the text of the recipe, for example "2 cloves garlic"
    food: Food                                     # one of the 78 foods of the table; any other name is refused
    grams: float                                   # the measure converted into grams


## Principles 1, 5: the calories of the question, with a strict input
@mcp.tool(annotations=READ_ONLY)
def recipe_count_calories(items: list[FoodWeight]) -> str:
    """Count the calories of one recipe.
    For each ingredient of the recipe, choose the closest food of the list and convert its measure into grams.
    Returns the calories of each ingredient and the total. The result is an estimate.

    Args:
        items: Each ingredient of the recipe: its text as written in the recipe, the closest food, and its weight in grams.
    """
    lines = []
    total = 0
    for item in items:
        kcal = KCAL[item.food] * item.grams / 100
        total = total + kcal
        lines.append("- " + item.ingredient + " -> " + item.food + ", " + str(round(item.grams)) + " g, " + str(round(kcal)) + " kcal")
    lines.append("Total: " + str(round(total)) + " kcal")
    return "\n".join(lines)


## Principles 1, 3: the shopping list of the question, as one list without what the user has
@mcp.tool(annotations=READ_ONLY)
def recipe_shopping_list(meal_ids: list[str], have: list[str]) -> str:
    """Make a shopping list for one or more meals: their ingredients without the ones that the user already has.

    Args:
        meal_ids: The IDs of the meals, for example ["52818", "53466"].
        have: The ingredients that the user has, in English, for example ["chicken breast", "garlic"].
    """
    to_buy = {}                                    # ingredient -> the measures for each meal
    for meal_id in meal_ids:
        meals = get_meals("/lookup.php", {"i": meal_id})
        if not meals:
            raise ToolError("No meal has the ID " + meal_id + ". Use an ID from recipe_find_by_ingredients.")
        meal = meals[0]
        for i in range(1, 21):
            name = meal["strIngredient" + str(i)]
            if not name:
                continue
            owned = False
            for item in have:
                if item.lower() in name.lower():   # "garlic" also matches "Garlic Clove"
                    owned = True
            if not owned:
                if name not in to_buy:
                    to_buy[name] = []
                to_buy[name].append(str(meal["strMeasure" + str(i)]) + " for " + meal["strMeal"])
    lines = ["To buy:"]
    for name in to_buy:
        lines.append("- " + name + " (" + "; ".join(to_buy[name]) + ")")
    return "\n".join(lines)


if __name__ == "__main__":
    mcp.run()

### 4.2 The same question on this server

*Do:* run the cell. It downloads the calorie table, connects to the second server, prints its tools, and runs `QUESTION` again.

In [ ]:
# ▶ RUN · example: read each line, then run
TABLE = "https://raw.githubusercontent.com/ralbu85/stml_2026/main/labs/data/food_calories.csv"
open("food_calories.csv", "w", encoding="utf-8").write(httpx2.get(TABLE).text)   # the server reads this file when it starts

server_v2 = PythonStdioTransport("meal_mcp_v2.py", log_file=Path("server.log"))
async with MCPAdapter(server_v2) as adapter:
    tools_v2 = await adapter.list_tools()

for tool in tools_v2:
    print(tool.name, list(tool.args))
print()

agent_v2 = create_agent(model, tools_v2)
result = await agent_v2.ainvoke({"messages": [HumanMessage(QUESTION)]})
show(result)

**Reading the output.** In our run the model made six calls. One call of `recipe_find_by_ingredients` found the meals; the model set `limit` to 2 because the question asks for two meals. It asked for the two recipes in one reply, then sent each recipe to `recipe_count_calories` as a list of items: the text of the recipe, a food of the table, and grams, for example \"3 diced chicken breast\" as 522 g of chicken breast. The last call made the shopping list. The longest tool result had 1,742 characters, against 42,111 in Part 2. The totals, about 2,134 and 3,939 kcal, are sums over the table; the grams in them are estimates of the model, and they can be read in the calls.

## 5. Exercises: Three More Tools

Each exercise adds one tool for a question that the five tools do not answer well. Write the tools in `meal_mcp_v3.py`. Its first lines import the server of Part 4, so your tools join its five tools. Follow the five principles: the prefix `recipe_`, a docstring that says when to use the tool, a result with only what the next step needs and a limit where it can be long, and a `ToolError` that says how to correct the call.

### 5.1 `recipe_find_by_country`

Question: 일본 요리 중에 닭가슴살이 들어가는 요리를 알려줘. The API returns the meals of a country with `GET /filter.php?a=<country>`, the country as an English adjective such as `Japanese`.

*Do:* write a tool that takes a country and, optionally, one ingredient, and returns the meals of that country that use the ingredient.

In [ ]:
%%writefile meal_mcp_v3.py
# ✍️ TRY · exercise: write your code in this cell, then run
from mcp.server.mcpserver.exceptions import ToolError
from meal_mcp_v2 import mcp, get_meals, READ_ONLY   # the server of Part 4 with its five tools, and its helpers

# recipe_find_by_country(country, ingredient, limit): the meals of one country, only those with the ingredient if one is given

### 5.2 `recipe_search_by_name`

Question: 카레 요리를 찾아줘. The API returns the meals whose name contains a word with `GET /search.php?s=<word>`, each with all 54 fields.

*Do:* write a tool that takes a word of a meal name and returns, for each meal, only its name, country, and ID, at most `limit` of them.

In [ ]:
%%writefile -a meal_mcp_v3.py
# ✍️ TRY · exercise: write your code in this cell, then run

# recipe_search_by_name(name, limit): the meals whose name contains the word, a few lines only

### 5.3 `recipe_check_ingredient`

Question: 대파랑 닭가슴살이 들어가는 요리 있어? The English name of 대파 is not clear, and the database knows 992 ingredient names, which `GET /list.php?i=list` returns.

*Do:* write a tool that takes an ingredient name and says whether the database knows it; if not, it suggests similar names that the database knows.

In [ ]:
%%writefile -a meal_mcp_v3.py
# ✍️ TRY · exercise: write your code in this cell, then run

# recipe_check_ingredient(name): is the name in the database? if not, which similar names are?

### 5.4 Your tools in the agent

*Do:* run the two cells. The first ends the file with the lines that run the server; the second connects to it, prints its tools, and runs the three questions.

In [ ]:
%%writefile -a meal_mcp_v3.py
# ▶ RUN · setup: just run


if __name__ == "__main__":
    mcp.run()

In [ ]:
# ▶ RUN · example: read each line, then run
server_v3 = PythonStdioTransport("meal_mcp_v3.py", log_file=Path("server.log"))
async with MCPAdapter(server_v3) as adapter:
    tools_v3 = await adapter.list_tools()

for tool in tools_v3:
    print(tool.name, list(tool.args))

agent_v3 = create_agent(model, tools_v3)
TESTS = [
    "일본 요리 중에 닭가슴살이 들어가는 요리를 알려줘.",
    "카레 요리를 찾아줘.",
    "대파랑 닭가슴살이 들어가는 요리 있어?",
]
for test in TESTS:
    print()
    print("=====", test)
    result = await agent_v3.ainvoke({"messages": [HumanMessage(test)]})
    show(result)